# Day 1 — Data Engineering

## 목차
1. [Databricks 개요](#1-databricks-개요)
2. [Databricks 핵심 구성요소](#2-databricks-핵심-구성요소)
3. [Data Engineering 심화 개념](#3-data-engineering-심화-개념)
4. [Databricks에서의 Data Engineering](#4-databricks에서의-data-engineering)
5. [Spark SQL vs RDB SQL](#5-spark-sql-vs-rdb-sql)
6. [Compute / Cluster 설정](#6-compute--cluster-설정)
7. [Hands-on Lab](#7-hands-on-lab)

---
## 1. Databricks 개요

- Databricks란 무엇인가
- 왜 사용하는가
- Lakehouse Architecture 개념
- Data Warehouse vs Data Lake vs Lakehouse
- Databricks Workspace 구조
  - Workspace / Cluster / Notebook / Job / Repo / SQL Warehouse

---
## 2. Databricks 핵심 구성요소

- **Delta Lake** — 데이터를 안전하게 저장하는 Databricks의 기본 파일 포맷
- **Unity Catalog** — 데이터 자산을 한 곳에서 통합 관리하는 거버넌스 시스템
- **Databricks Runtime** — 클러스터 실행 환경 (Spark + 라이브러리 묶음)
- **Medallion Architecture** — 데이터 품질 단계별 분리 (Bronze → Silver → Gold)

---
## 3. Data Engineering 심화 개념

- **ETL 중심으로의 전환** — 왜 Databricks에서는 ETL이 표준인가
- **Batch vs Micro-batch vs Streaming** — 선택 기준과 트레이드오프
- **파일 포맷 선택 전략** — Parquet vs Delta, 언제 무엇을 쓰는가
- **파티셔닝 전략** — 과도한 파티셔닝의 문제 (Small File Problem)
- **데이터 파이프라인 설계 원칙** — 멱등성(Idempotency), 재처리 가능성
- **Medallion Architecture 심화** — Bronze/Silver/Gold 경계 기준과 실무 적용

---
## 4. Databricks에서의 Data Engineering

- **Notebook 기반 개발 방식**
- **Spark 기본 개념** — Driver / Executor / Distributed Processing
- **DataFrame 기본 사용** / Spark SQL 기본
- **AWS S3 연동** 및 데이터 Ingest 개념
- **Delta Table 생성 및 관리**

---
## 5. Spark SQL vs RDB SQL

### SQL 문법은 같지만 실행 방식이 다르다

| 구분 | RDB | Spark SQL |
|------|-----|-----------|
| 실행 방식 | 단일 서버, 인덱스 기반 행 탐색 | 분산 클러스터, 파일 전체 병렬 스캔 |
| 인덱스 | O | X (파티셔닝 + 파일 포맷으로 대체) |
| 트랜잭션 | Row-level lock, OLTP 최적화 | MVCC 방식, 대용량 배치 최적화 |
| JOIN | Nested Loop / Hash Join | Shuffle Hash Join / Broadcast Join |

### 지원하지 않는 것들
- Row-level UPDATE/DELETE는 Delta Lake 없이 불가
- Stored Procedure / Trigger 미지원
- Auto Increment PK 개념 없음

### Spark SQL만의 강점
- 수십억 건 데이터도 수분 내 집계 가능
- DataFrame API와 혼용 가능 (SQL ↔ Python 전환 자유로움)
- `%sql` 매직 커맨드로 Notebook 내 SQL 직접 실행

---
## 6. Compute / Cluster 설정

> 실습 전 직접 클러스터를 생성하고 설정하는 과정을 다룬다

### Compute 유형
| 유형 | 용도 |
|------|------|
| All-purpose Cluster | 개발/탐색용, 대화형 노트북 실행 |
| Job Cluster | 자동화된 Job 실행 전용, 실행 후 자동 종료 |
| SQL Warehouse | SQL 분석 전용 컴퓨팅 |
| Serverless Compute | 인프라 관리 없이 즉시 사용 |

### Cluster 생성
- Cluster 이름 / Access Mode 설정
- Databricks Runtime 버전 선택 기준
- Worker / Driver Node 타입 및 수량 설정
- Auto Scaling 설정 — 최소/최대 Worker 수
- Auto Termination 설정 — 유휴 시 자동 종료
- Spot Instance vs On-demand 차이

### Cluster 고급 설정
- Spark Config 추가 (환경변수, 튜닝 파라미터)
- Init Script — 클러스터 시작 시 자동 실행 스크립트
- 라이브러리 설치 (PyPI / Maven / DBFS)
- Cluster Policy — 조직 내 사용 규칙 적용
- Cluster Tags — 비용 추적을 위한 태그 설정

### Cluster 운영
- Cluster 상태 확인 (Running / Terminated / Pending)
- Cluster Event Log — 이벤트 및 오류 확인
- Spark UI — Job / Stage / Task 모니터링

### Notebook과 Cluster 연결
- Notebook에 Cluster 연결하는 방법
- 여러 Notebook에서 동일 Cluster 공유
- Detach / Reattach 시 주의사항

---
## 7. Hands-on Lab

### Step 1. 환경 확인
Spark 버전 및 현재 카탈로그/스키마 확인

In [ ]:
print(f"Spark Version: {spark.version}")
print(f"현재 카탈로그: {spark.catalog.currentCatalog()}")
print(f"현재 스키마: {spark.catalog.currentDatabase()}")
print(f"현재 사용자: {spark.sql('SELECT current_user()').collect()[0][0]}")

### Step 2. 카탈로그 / 스키마 설정

In [ ]:
# 카탈로그 및 스키마 변경
spark.sql("USE CATALOG <카탈로그명>")
spark.sql("USE <스키마명>")

### Step 3. CSV 데이터 로드

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

file_path = "/Volumes/<카탈로그>/<스키마>/<볼륨>/<파일명>.csv"

# inferSchema 방식
df = spark.read.option("header", True).option("inferSchema", True).csv(file_path)

# 명시적 스키마 방식
schema = StructType([
    StructField("id",      IntegerType(), True),
    StructField("name",    StringType(),  True),
    StructField("value",   DoubleType(),  True),
])
df = spark.read.option("header", True).schema(schema).csv(file_path)

display(df)

### Step 4. DataFrame 변환

In [ ]:
from pyspark.sql.functions import col, when, count, avg, round

# select
df_selected = df.select("id", "name", "value")

# filter
df_filtered = df.filter(col("value") >= 100)

# withColumn
df_grade = df.withColumn(
    "grade",
    when(col("value") >= 200, "high")
    .when(col("value") >= 100, "mid")
    .otherwise("low")
)

# groupBy + agg
df_agg = df.groupBy("name").agg(
    count("*").alias("cnt"),
    round(avg("value"), 1).alias("avg_value")
).orderBy("cnt", ascending=False)

display(df_agg)

### Step 5. Spark SQL

In [ ]:
# Temp View 등록
df.createOrReplaceTempView("raw_data")

```sql
%sql
SELECT
    name,
    COUNT(*) AS cnt,
    ROUND(AVG(value), 1) AS avg_value
FROM raw_data
GROUP BY name
ORDER BY cnt DESC
```

### Step 6. Delta Table 생성 및 CRUD

In [ ]:
# Delta Table 생성
df.write.mode("overwrite").saveAsTable("<카탈로그>.<스키마>.bronze_table")

```sql
%sql
-- INSERT
INSERT INTO <카탈로그>.<스키마>.bronze_table VALUES (999, 'test', 50.0);

-- UPDATE
UPDATE <카탈로그>.<스키마>.bronze_table SET value = value + 10 WHERE name = 'test';

-- DELETE
DELETE FROM <카탈로그>.<스키마>.bronze_table WHERE id = 999;

-- MERGE (Upsert)
MERGE INTO <카탈로그>.<스키마>.bronze_table AS target
USING (SELECT 1 AS id, 'new' AS name, 99.0 AS value) AS source
ON target.id = source.id
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

### Step 7. Time Travel

```sql
%sql
-- 변경 이력 확인
DESCRIBE HISTORY <카탈로그>.<스키마>.bronze_table;

-- 특정 버전 조회
SELECT * FROM <카탈로그>.<스키마>.bronze_table VERSION AS OF 0;

-- 특정 버전으로 복원
RESTORE TABLE <카탈로그>.<스키마>.bronze_table VERSION AS OF 0;
```

### Step 8. Bronze → Silver 파이프라인

In [ ]:
# Bronze 테이블 로드
df_bronze = spark.table("<카탈로그>.<스키마>.bronze_table")

# 데이터 정제 (Silver 변환)
df_silver = (
    df_bronze
    .filter(col("value").isNotNull())
    .withColumn("grade",
        when(col("value") >= 200, "high")
        .when(col("value") >= 100, "mid")
        .otherwise("low")
    )
)

# Silver 테이블 저장
df_silver.write.mode("overwrite").saveAsTable("<카탈로그>.<스키마>.silver_table")

print("Bronze → Silver 파이프라인 완료")
display(spark.table("<카탈로그>.<스키마>.silver_table"))